# 🔧 2_Transformacion — ETL Banano Ecuador

**Tarea del Job:** `tarea_transformacion` (se ejecuta solo si `hay_nuevos == "true"`)

Lee los archivos crudos descargados por `1_Extraccion`, aplica la transformación
especializada según la fuente (ESPAC, SIPA, FAOSTAT, AEBE), mapea cada archivo a
su tabla destino (LLM + reglas) y carga los datos limpios en las tablas Delta de
`bd_banano_ec`.

**Salidas:**
- Tablas dimensionales `bd_banano_ec.dim_regiones`, `bd_banano_ec.dim_provincias`
- Tablas de datos: `espac_banano_platano_provincia`, `espac_uso_del_suelo`,
  `sipa_temperatura_precipitacion`, `faostat_produccion_banano_platano`,
  `aebe_exportaciones_regiones`
- Task Values: `status`, `registros_procesados` (usados por la condición del DAG)


## Instalación de librerías
⚠️ Ejecutar solo esta celda primero, esperar el reinicio del kernel.

In [ ]:
%pip install langchain langchain-databricks openpyxl xlrd numpy scikit-learn pdfplumber
dbutils.library.restartPython()


## Configuración e inicialización (LLM, rutas, base de datos)

In [ ]:
import io, os, re, json, unicodedata
import numpy as np
import pandas as pd
from datetime import datetime
from typing import TypedDict, Optional, List

# LangChain + Databricks Foundation Models
from langchain_databricks import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import trim, col, when
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

# ── Rutas de volúmenes Databricks (origen de los archivos crudos) ─────────
RAW_PATH_ESPAC   = "/Volumes/workspace/default/raw_espac"
RAW_PATH_SIPA    = "/Volumes/workspace/default/raw_sipa"
RAW_PATH_FAOSTAT = "/Volumes/workspace/default/raw_faostat"
RAW_PATH_AEBE    = "/Volumes/workspace/default/raw_aebe"

# ── Configuración de base de datos ─────────────────────────────────────────
DB_NAME = "bd_banano_ec"

# Constante de compatibilidad: el agente siempre opera en modo normal.
ESCENARIO = "A"

# ── LLM (Databricks Foundation Models) ────────────────────────────────────
llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",  # Llama 4 Maverick - 400B MoE, 128K context
    temperature=0.0,
    max_tokens=2000,
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
print("✅ Configuración cargada correctamente.")
print(f"   Base de datos: {DB_NAME}")


## Diccionario de conocimiento (mapeo archivo → tabla Delta) y tabla AEBE

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DICCIONARIO DE CONOCIMIENTO CORREGIDO
# Mapeo archivo → tabla Delta (usado por el LLM para decidir destino)
# ═══════════════════════════════════════════════════════════════════════════

DICCIONARIO_CONOCIMIENTO = """
{
  "mapeo_archivos": [
    {"archivo": "ESPAC_T13",                    "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_T26",                    "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_Tabulados_excel",        "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_Series_historicas",      "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_USO_DEL_SUELO",          "tabla_destino": "espac_uso_del_suelo",            "fuente": "INEC_ESPAC"},
    {"archivo": "SIPA_USO_SUELO",               "tabla_destino": "espac_uso_del_suelo",            "fuente": "SIPA_MAG"},
    {"archivo": "TEMPERATURA_Y_PRECIPITACION",  "tabla_destino": "sipa_temperatura_precipitacion", "fuente": "SIPA_MAG"},
    {"archivo": "SIPA_TEMPERATURA",             "tabla_destino": "sipa_temperatura_precipitacion", "fuente": "SIPA_MAG"},
    {"archivo": "FAOSTAT",                      "tabla_destino": "faostat_produccion_banano_platano", "fuente": "FAOSTAT"},
    {"archivo": "AEBE_BANANOTAS",                "tabla_destino": "aebe_exportaciones_regiones",   "fuente": "AEBE_BANANOTAS", "descripcion": "Rankings consolidados: exportadores, marcas, navieras, puertos, mercados. Año extraído del contenido (max year en texto). Estructura: tipo|dato|cantidad|medida|anio|framework|fuente|archivo_origen"}
  ]
}
"""

# ── Crear/verificar tabla AEBE con schema correcto ─────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.aebe_exportaciones_regiones (
    tipo              STRING COMMENT 'Tipo de ranking: exportador, marca, naviera, puerto, mercado',
    dato              STRING COMMENT 'Nombre de la entidad (empresa, marca, puerto, país)',
    cantidad          DOUBLE COMMENT 'Valor para el año detectado (cajas en millones)',
    medida            STRING COMMENT 'Unidad de medida (ej: cajas (millones))',
    anio              INT    COMMENT 'Año del dato (extraído del documento, siempre el mayor)',
    framework         STRING COMMENT 'Framework ETL usado: AgenteCustom',
    fuente            STRING COMMENT 'Fuente original: AEBE_BANANOTAS',
    archivo_origen    STRING COMMENT 'Nombre del archivo PDF de origen'
) USING DELTA
""")

print("✅ Diccionario de conocimiento CORREGIDO cargado.")
print("   • ESPAC_Tabulados_excel y ESPAC_Series_historicas → espac_banano_platano_provincia")
print("   • FAOSTAT → faostat_produccion_banano_platano")
print("   • AEBE_BANANOTAS → aebe_exportaciones_regiones")
print("   • SIPA_TEMPERATURA → sipa_temperatura_precipitacion")

## Tablas dimensionales: regiones y provincias de Ecuador

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLAS DIMENSIONALES: REGIONES Y PROVINCIAS DE ECUADOR
# Esta estructura maestra normaliza los datos geográficos
# ══════════════════════════════════════════════════════════════════════════

print("📍 Creando/verificando tablas dimensionales...\n")

# ──────────────────────────────────────────────────────────────────────────
# 1️⃣ TABLA DIMENSIONAL: REGIONES
# ──────────────────────────────────────────────────────────────────────────
tabla_dim_regiones = f"{DB_NAME}.dim_regiones"

if spark.catalog.tableExists(tabla_dim_regiones):
    print(f"🌍 1. Tabla {tabla_dim_regiones} ya existe")
    count_regiones = spark.table(tabla_dim_regiones).count()
    print(f"   ℹ️  {count_regiones} regiones registradas (sin cambios)\n")
else:
    print("🌍 1. Creando dim_regiones...")
    
    regiones_data = [
        (1, "Sierra", "Región Interandina"),
        (2, "Costa", "Región Litoral"),
        (3, "Amazonía", "Región Amazónica"),
        (4, "Insular", "Región Insular - Galápagos"),
    ]
    
    schema_regiones = StructType([
        StructField("region_id", IntegerType(), False),
        StructField("region", StringType(), False),
        StructField("descripcion", StringType(), True),
    ])
    
    df_regiones = spark.createDataFrame(regiones_data, schema=schema_regiones)
    df_regiones.write.mode("overwrite").saveAsTable(tabla_dim_regiones)
    
    print(f"   ✅ Tabla creada: {tabla_dim_regiones}")
    print(f"   📊 {df_regiones.count()} regiones registradas\n")

# ──────────────────────────────────────────────────────────────────────────
# 2️⃣ TABLA DIMENSIONAL: PROVINCIAS (con FK a regiones)
# ──────────────────────────────────────────────────────────────────────────
tabla_dim_provincias = f"{DB_NAME}.dim_provincias"

if spark.catalog.tableExists(tabla_dim_provincias):
    print(f"🗺️  2. Tabla {tabla_dim_provincias} ya existe")
    count_provincias = spark.table(tabla_dim_provincias).count()
    print(f"   ℹ️  {count_provincias} provincias registradas (sin cambios)\n")
else:
    print("🗺️  2. Creando dim_provincias...")
    
    provincias_data = [
        (1, "Azuay", 1),
        (2, "Bolívar", 1),
        (3, "Cañar", 1),
        (4, "Carchi", 1),
        (5, "Cotopaxi", 1),
        (6, "Chimborazo", 1),
        (7, "El Oro", 2),
        (8, "Esmeraldas", 2),
        (9, "Guayas", 2),
        (10, "Imbabura", 1),
        (11, "Loja", 1),
        (12, "Los Ríos", 2),
        (13, "Manabí", 2),
        (14, "Morona Santiago", 3),
        (15, "Napo", 3),
        (16, "Pastaza", 3),
        (17, "Pichincha", 1),
        (18, "Tungurahua", 1),
        (19, "Zamora Chinchipe", 3),
        (20, "Galápagos", 4),
        (21, "Sucumbíos", 3),
        (22, "Orellana", 3),
        (23, "Santo Domingo De Los Tsáchilas", 1),
        (24, "Santa Elena", 2),
    ]
    
    schema_provincias = StructType([
        StructField("provincia_id", IntegerType(), False),
        StructField("provincia", StringType(), False),
        StructField("region_id", IntegerType(), False),
    ])
    
    df_provincias = spark.createDataFrame(provincias_data, schema=schema_provincias)
    
    # Agregar variantes normalizadas para el mapeo (sin tildes, minúsculas)
    df_provincias = df_provincias.withColumn(
        "provincia_normalizada",
        F.lower(F.translate(F.col("provincia"), "áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnn"))
    )
    
    df_provincias.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla_dim_provincias)
    
    print(f"   ✅ Tabla creada: {tabla_dim_provincias}")
    print(f"   📊 {df_provincias.count()} provincias registradas\n")

# ──────────────────────────────────────────────────────────────────────────
# VISTA PREVIA
# ──────────────────────────────────────────────────────────────────────────
print("🔍 Vista previa de la estructura dimensional:\n")

print("📊 Consulta unificada (con JOIN):")
spark.sql(f"""
SELECT 
    p.provincia_id,
    p.provincia,
    r.region_id,
    r.region,
    r.descripcion AS region_descripcion
FROM {tabla_dim_provincias} p
JOIN {tabla_dim_regiones} r ON p.region_id = r.region_id
ORDER BY r.region_id, p.provincia_id
""").display()

print("\n✅ Estructura dimensional creada exitosamente!")
print("\n📝 Resumen:")
print("   • 4 regiones (Sierra, Costa, Amazonía, Insular)")
print("   • 24 provincias con FK a regiones")
print("   • provincia_normalizada para mapeo automático")

## Mapeo de provincia → provincia_id

In [ ]:
def mapear_provincia_a_id(df_pandas: pd.DataFrame, columna_provincia: str = 'provincia') -> pd.DataFrame:
    """
    Mapea nombres de provincias a sus IDs usando la tabla dimensional.
    
    Args:
        df_pandas: DataFrame con datos que incluyen nombres de provincias
        columna_provincia: Nombre de la columna que contiene el nombre de provincia
    
    Returns:
        DataFrame con provincia_id en lugar de nombre de provincia
    """
    if columna_provincia not in df_pandas.columns:
        print(f"  ⚠️  Columna '{columna_provincia}' no encontrada, omitiendo mapeo")
        return df_pandas
    
    print(f"  🗺️  Mapeando provincias a IDs...")
    
    # Leer tabla dimensional como pandas para el mapeo
    dim_prov_pd = spark.table(f"{DB_NAME}.dim_provincias").toPandas()
    
    # Crear diccionario de mapeo (provincia_normalizada -> provincia_id)
    mapeo_dict = dict(zip(dim_prov_pd['provincia_normalizada'], dim_prov_pd['provincia_id']))
    
    # Normalizar la columna de provincia en el DataFrame de datos
    def normalizar_provincia(texto):
        if pd.isna(texto) or texto == '':
            return None
        # Convertir a string, quitar espacios, minusculas
        texto = str(texto).strip().lower()
        # Quitar tildes
        texto = texto.translate(str.maketrans('áéíóúÁÉÍÓÚñÑ', 'aeiouAEIOUnn'))
        return texto
    
    df_pandas['_provincia_norm'] = df_pandas[columna_provincia].apply(normalizar_provincia)
    
    # Mapear a provincia_id
    df_pandas['provincia_id'] = df_pandas['_provincia_norm'].map(mapeo_dict)
    
    # Verificar cuántas NO mapearon
    sin_mapear = df_pandas['provincia_id'].isna().sum()
    if sin_mapear > 0:
        print(f"    ⚠️  {sin_mapear} registros sin mapeo de provincia")
        provincias_no_mapeadas = df_pandas[df_pandas['provincia_id'].isna()][columna_provincia].unique()
        print(f"       Provincias no reconocidas: {list(provincias_no_mapeadas)[:5]}")
    
    # Eliminar columnas auxiliares y original
    df_pandas = df_pandas.drop(columns=['_provincia_norm', columna_provincia])
    
    # Convertir provincia_id a int (donde no sea null)
    df_pandas['provincia_id'] = df_pandas['provincia_id'].astype('Int64')
    
    mapeados = (~df_pandas['provincia_id'].isna()).sum()
    print(f"    ✅ {mapeados} provincias mapeadas correctamente")
    
    return df_pandas

print("✅ Función mapear_provincia_a_id() definida.")

## Funciones utilitarias compartidas (normalización, detección de header, lectura de archivos)

In [ ]:
def normalizar_columna(col_name: str) -> str:
    c = str(col_name).lower().strip()
    c = c.replace(" ","_").replace("-","_")
    c = ''.join(ch for ch in unicodedata.normalize('NFD', c) if unicodedata.category(ch) != 'Mn')
    c = re.sub(r'[^a-z0-9_]', '', c)
    c = re.sub(r'_+', '_', c).strip('_')
    return c if c else "columna_sin_nombre"

def identificar_fuente(file_name: str) -> str:
    n = file_name.upper()
    if n.startswith("ESPAC_"):         return "INEC_ESPAC"
    if n.startswith("TEMPERATURA_"):   return "SIPA_MAG"
    if n.startswith("SIPA_"):          return "SIPA_MAG"
    if n.startswith("FAOSTAT_"):       return "FAOSTAT"
    if n.startswith("AEBE_"):          return "AEBE_BANANOTAS"
    return "DESCONOCIDA"

def _aplanar_excel_openpyxl(local_path: str) -> pd.DataFrame:
    """Expande celdas combinadas antes de leer — necesario para archivos ESPAC."""
    from openpyxl import load_workbook
    wb = load_workbook(local_path, data_only=True)
    ws = wb.active
    for mr in list(ws.merged_cells.ranges):
        val = ws.cell(mr.min_row, mr.min_col).value
        ws.unmerge_cells(str(mr))
        for r in range(mr.min_row, mr.max_row + 1):
            for c in range(mr.min_col, mr.max_col + 1):
                ws.cell(r, c).value = val
    data = [list(row) for row in ws.iter_rows(values_only=True)]
    wb.close()
    if not data: return pd.DataFrame()
    max_cols = max(len(r) for r in data)
    for r in data:
        while len(r) < max_cols: r.append(None)
    df = pd.DataFrame(data)
    return df.astype(str).replace({"None":"","nan":"","NaT":""})

def _score_header(raw_df: pd.DataFrame, row_idx: int) -> int:
    """Puntúa la probabilidad de que una fila sea el encabezado real de la tabla."""
    if row_idx >= len(raw_df): return -99
    vals  = [str(v).strip() for v in raw_df.iloc[row_idx].values]
    score = 0
    ratio = sum(1 for v in vals if v == "") / max(len(vals), 1)
    score += 3 if ratio == 0 else (1 if ratio <= 0.25 else -2)
    KEYWORDS = ["áño","anio","year","provincia","producto","cultivo","superficie",
                "area","produccion","tonelada","rendimiento","precio","valor","zona"]
    row_text = " ".join(vals).lower()
    for a, b in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        row_text = row_text.replace(a, b)
    score += min(sum(1 for kw in KEYWORDS if kw in row_text) * 2, 10)
    n_etiq = sum(1 for v in vals if v and len(v)<=50 and not
                 v.replace(".","").replace(",","").replace("-","").replace(" ","").isdigit())
    if n_etiq / max(len(vals),1) >= 0.65: score += 2
    if row_idx+1 < len(raw_df):
        nxt   = [str(v).strip() for v in raw_df.iloc[row_idx+1].values]
        n_num = sum(1 for v in nxt if v and v.replace(".","").replace(",","").replace("-","").isdigit())
        if n_num / max(len(nxt),1) >= 0.25: score += 4
    if any(re.search(p, row_text) for p in [r"fuente\s*:",r"nota\s*:",r"ministerio",r"encuesta",r"http"]):
        score -= 4
    return score

def leer_archivo(local_path: str, file_name: str) -> pd.DataFrame:
    """
    Lee el archivo y retorna un DataFrame con columnas normalizadas y header detectado.
    CORRECCIÓN #5 MEJORADO: Usa detección automática limitada para SIPA en lugar de hardcodear filas.
    """
    ext = file_name.lower()
    
    # ── PDF (AEBE BANANOTAS) ──────────────────────────────
    # Los PDFs se manejan directamente en transformar_aebe_bananotas
    if ext.endswith(".pdf"):
        print(f"  📝 PDF detectado - se procesará con transformación especializada")
        return pd.DataFrame({"_es_pdf": [True]})  # Placeholder
    
    # ── CORRECCIÓN #5 MEJORADO: SIPA TEMPERATURA/PRECIPITACIÓN ──────────────
    if "SIPA_TEMPERATURA" in file_name.upper():
        print(f"  🔧 SIPA Temperatura: Detección automática de header (limitada a primeras 10 filas)")
        df_raw = pd.read_excel(local_path, header=None, dtype=str, engine="xlrd")
        
        # Buscar header solo en primeras 10 filas
        scores = {i: _score_header(df_raw, i) for i in range(min(10, len(df_raw)))}
        mejor = max(scores, key=scores.get)
        print(f"    ✅ Header detectado en fila {mejor} (score: {scores[mejor]})")
        
        # Extraer header y datos
        header_vals = list(df_raw.iloc[mejor].values)
        col_names = [normalizar_columna(str(v)) if str(v).strip() else f"col_{i}" for i, v in enumerate(header_vals)]
        
        df = df_raw.iloc[mejor+1:].copy()
        df.columns = col_names
        df = df.reset_index(drop=True)
        
        # Limpiar SOLO filas completamente vacías (mejora clave: no usar .dropna(how='all') que es demasiado agresivo)
        df = df.replace(r'^\s*$', np.nan, regex=True)  # Convertir strings vacíos a NaN
        df = df[df.notna().any(axis=1)]  # Filtrar filas donde TODAS las columnas son NaN
        
        print(f"    📊 Registros después de limpieza: {len(df):,}")
        return df.reset_index(drop=True)
    
    # ── CORRECCIÓN #5 MEJORADO: SIPA USO DEL SUELO ──────────────────────────
    if "SIPA_USO" in file_name.upper() or "ESPAC_SIPA_USO" in file_name.upper():
        print(f"  🔧 SIPA Uso del Suelo: Detección automática de header (limitada a primeras 10 filas)")
        df_raw = pd.read_excel(local_path, header=None, dtype=str, engine="openpyxl")
        
        # Buscar header solo en primeras 10 filas
        scores = {i: _score_header(df_raw, i) for i in range(min(10, len(df_raw)))}
        mejor = max(scores, key=scores.get)
        print(f"    ✅ Header detectado en fila {mejor} (score: {scores[mejor]})")
        
        # Extraer header y datos
        header_vals = list(df_raw.iloc[mejor].values)
        col_names = [normalizar_columna(str(v)) if str(v).strip() else f"col_{i}" for i, v in enumerate(header_vals)]
        
        df = df_raw.iloc[mejor+1:].copy()
        df.columns = col_names
        df = df.reset_index(drop=True)
        
        # Limpiar SOLO filas completamente vacías
        df = df.replace(r'^\s*$', np.nan, regex=True)
        df = df[df.notna().any(axis=1)]
        
        print(f"    📊 Registros después de limpieza: {len(df):,}")
        return df.reset_index(drop=True)
    
    # ── JSON (FAOSTAT) ─────────────────────────────────────────────────
    if ext.endswith(".json"):
        with open(local_path, 'r') as f:
            data = json.load(f)
        
        if isinstance(data, dict) and 'data' in data:
            records = data['data']
        elif isinstance(data, list):
            records = data
        else:
            raise ValueError(f"Formato JSON no reconocido: {file_name}")
        
        if not records:
            return pd.DataFrame()
        
        df = pd.DataFrame(records)
        df.columns = [normalizar_columna(c) for c in df.columns]
        return df
    
    # ── EXCEL ────────────────────────────────────────────────────────
    elif ext.endswith(".xls"):
        raw = pd.read_excel(local_path, header=None, dtype=str, engine="xlrd")
    elif ext.endswith(".xlsx"):
        try:   raw = _aplanar_excel_openpyxl(local_path)
        except: raw = pd.read_excel(local_path, header=None, dtype=str, engine="openpyxl")
    
    # ── CSV ───────────────────────────────────────────────────────────
    elif ext.endswith(".csv"):
        sep = ';' if any(x in file_name.upper() for x in ['T13', 'T26']) else ','
        
        for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
            try:
                raw = pd.read_csv(local_path, header=None, dtype=str, encoding=encoding, sep=sep, on_bad_lines='skip')
                break
            except (UnicodeDecodeError, LookupError):
                continue
        else:
            raw = pd.read_csv(local_path, header=None, dtype=str, encoding='latin1', errors='ignore', sep=sep, on_bad_lines='skip')
    else:
        raise ValueError(f"Formato no soportado: {file_name}")

    # ── DETECCIÓN AUTOMÁTICA DE HEADER (para Excel/CSV no-SIPA) ───────────────────
    raw = raw.astype(str).replace({"None":"","nan":"","NaT":""})
    scores = {i: _score_header(raw, i) for i in range(min(40, len(raw)))}
    mejor  = max(scores, key=scores.get)
    if scores[mejor] < 3: mejor = 0

    header_vals = list(raw.iloc[mejor].values)
    seen, col_names = {}, []
    for v in header_vals:
        v = normalizar_columna(str(v)) if str(v).strip() else "col_sin_nombre"
        seen[v] = seen.get(v,0) + 1
        col_names.append(f"{v}_{seen[v]-1}" if seen[v]>1 else v)

    df = raw.iloc[mejor+1:].copy()
    df.columns = col_names
    df = df.reset_index(drop=True)
    df = df.replace(r'^\s*[\.-]*\s*$', np.nan, regex=True)
    df = df.dropna(how="all")  # Esto es OK para archivos no-SIPA
    return df

def castear_columnas(df_spark, cols_double, cols_integer):
    """Castea columnas numéricas tolerando comas como separador decimal."""
    PROTEGIDAS = {"mes","meses","ano","áño","provincia","canton"}
    for c in cols_double:
        if c in df_spark.columns and c not in PROTEGIDAS:
            df_spark = (df_spark
                .withColumn(c, F.regexp_replace(F.col(c), r'\s+', ''))
                .withColumn(c, F.regexp_replace(F.col(c), ',', '.'))
                .withColumn(c, F.when(F.col(c)=='.', None).otherwise(F.col(c)))
                .withColumn(c, F.expr(f"try_cast(`{c}` as double)")))
    for c in cols_integer:
        if c in df_spark.columns and c not in PROTEGIDAS:
            df_spark = (df_spark
                .withColumn(c, F.regexp_replace(F.col(c), r'\..*', ''))
                .withColumn(c, F.regexp_replace(F.col(c), r'\s+', ''))
                .withColumn(c, F.when(F.col(c)=='.', None).otherwise(F.col(c)))
                .withColumn(c, F.expr(f"try_cast(`{c}` as integer)")))
    return df_spark

print("✅ Funciones utilitarias cargadas.")

## Transformación especializada: AEBE Bananotas y ESPAC (múltiples hojas)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRANSFORMACIONES ESPECIALIZADAS PARA ARCHIVOS COMPLEJOS
# ══════════════════════════════════════════════════════════════════════════════

def transformar_aebe_bananotas(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma PDFs de AEBE Bananotas extrayendo rankings de:
    - Exportadores (Ubesa, Reybanpac, etc.)
    - Marcas (Dole, Chiquita, etc.)
    - Navieras (MSC, Maersk, etc.)
    - Puertos (San Petersburgo, Rotterdam, etc.)
    - Mercados/países (Unión Europea, Medio Oriente, etc.)

    MOTOR: pdfplumber (puro Python, sin JVM) en lugar de tabula-py.
    tabula-py lanza un subproceso JVM por cada llamada a read_pdf(), y el
    código anterior lo invocaba ~20 veces por PDF (5 tipos x ~4 páginas),
    lo que multiplicaba el overhead de arranque de JVM y explicaba los
    tiempos de horas. pdfplumber abre el PDF una sola vez y recorre todas
    las páginas en una sola pasada en memoria.

    DETECCIÓN DINÁMICA: no se asume un rango fijo de páginas (ej. 65-76,
    que puede variar de edición a edición). En su lugar, se inspecciona el
    texto de CADA página del PDF buscando tablas cuya fila de cabecera
    contenga dos columnas de año consecutivas (ej. "2025" y "2026") como
    valores aislados — patrón característico de las tablas de ranking de
    Bananotas. El tipo (exportador/marca/naviera/puerto/mercado) se infiere
    por palabras clave en el título inmediatamente anterior a la tabla, con
    fallback a la propia fila de cabecera.

    NORMALIZACIÓN A UN SOLO AÑO: cada tabla de ranking trae dos columnas de
    cantidad (año N y año N+1, ej. 2025 y 2026). Se conserva únicamente la
    columna del año MÁS RECIENTE de cada tabla — no se asume que todas las
    tablas del PDF compartan el mismo par de años, por lo que el año
    "más reciente" se calcula tabla por tabla, y el año de la revista
    completa es el máximo visto en todas las tablas.

    Estructura normalizada (UN registro por entidad del ranking):
    tipo | dato | cantidad | medida | anio | framework | fuente | archivo_origen

    Args:
        local_path: Ruta local al PDF descargado
        nombre_archivo: Nombre del archivo

    Returns:
        DataFrame consolidado con todos los rankings del año más reciente
        detectado en cada tabla.
    """
    print(f"  📊 Transformación AEBE Bananotas (pdfplumber): {nombre_archivo}")

    try:
        import pdfplumber
    except ImportError:
        print("  ⚠️  Instalando pdfplumber...")
        import subprocess
        subprocess.run(["pip", "install", "-q", "pdfplumber"], check=True)
        import pdfplumber

    COLUMNAS_VACIAS = ['tipo', 'dato', 'cantidad', 'medida', 'anio', 'framework', 'fuente', 'archivo_origen']

    RANKING_TIPOS = {
        'exportador': ['EXPORTADOR', 'EXPORTADORES'],
        'marca':      ['MARCA', 'MARCAS'],
        'naviera':    ['NAVIERA', 'NAVIERAS'],
        'puerto':     ['PUERTO', 'PUERTOS', 'DESTINO'],
        'mercado':    ['PAIS', 'PAÍS', 'MERCADO', 'MERCADOS'],
    }

    def _detectar_anios_header(tabla_filas):
        """Fila de cabecera real = la que contiene >=2 años de 4 dígitos.
        Busca años dentro de las celdas (no solo celdas que sean exactamente un año),
        para manejar headers con formato 'R MARCAS 2025 2026\n2025 2026...'."""
        for fila in tabla_filas:
            celdas = [str(c).strip() if c else "" for c in fila]
            # Buscar TODOS los años en TODAS las celdas (incluso si hay otros textos)
            anios_encontrados = []
            for celda in celdas:
                # Encuentra todos los años de 4 dígitos en esta celda
                matches = re.findall(r'\b20[12][0-9]\b', celda)
                anios_encontrados.extend(matches)
            
            # Si encontramos al menos 2 años distintos, es el header
            anios_unicos = sorted(set(int(a) for a in anios_encontrados))
            if len(anios_unicos) >= 2:
                return anios_unicos, fila
        return None, None

    def _clasificar_tipo(texto):
        texto_up = texto.upper()
        for tipo, keywords in RANKING_TIPOS.items():
            if any(kw in texto_up for kw in keywords):
                return tipo
        return None

    def _limpiar_numero(val):
        if val is None:
            return None
        s = str(val).strip()
        if s in ("", "-", "—", "–"):
            return None
        s = s.replace("%", "")
        s = re.sub(r"\s+", "", s)
        if "," in s and "." in s:
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None

    registros = []
    anios_vistos = []

    try:
        with pdfplumber.open(local_path) as pdf:
            print(f"    🔍 Escaneando {len(pdf.pages)} páginas (parseo de texto)...")
            
            for page_num, page in enumerate(pdf.pages, start=1):
                texto = page.extract_text()
                if not texto:
                    continue
                
                lineas = texto.split('\n')
                
                # Buscar líneas con header (2 años consecutivos)
                for idx, linea in enumerate(lineas):
                    # Detectar header con años
                    anios = re.findall(r'\b20[12][0-9]\b', linea)
                    if len(set(anios)) < 2:
                        continue
                    
                    anios_unicos = sorted(set(int(a) for a in anios))
                    anio_reciente = max(anios_unicos)
                    anios_vistos.append(anio_reciente)
                    
                    # Determinar tipo de ranking
                    tipo = None
                    contexto = ' '.join(lineas[max(0, idx-3):idx+1]).upper()
                    tipo = _clasificar_tipo(contexto)
                    
                    if not tipo:
                        continue
                    
                    # Parsear filas de datos después del header
                    n_antes = len(registros)
                    for linea_datos in lineas[idx+1:idx+25]:  # Máximo 25 líneas después
                        # Patrón: ranking + nombre + números
                        match = re.match(r'^(\d{1,2})\s+(.+?)\s+([\d.,]+)\s+([\d.,]+)', linea_datos)
                        if not match:
                            continue
                        
                        nombre = match.group(2).strip()
                        
                        if nombre.upper() in ['TOTAL', 'OTROS', 'TOTAL GENERAL']:
                            continue
                        
                        valor1 = _limpiar_numero(match.group(3))
                        valor2 = _limpiar_numero(match.group(4))
                        
                        # Segundo valor = año más reciente
                        cantidad = valor2 if valor2 else valor1
                        
                        if not cantidad or cantidad <= 0:
                            continue
                        
                        registros.append({
                            'tipo': tipo,
                            'dato': nombre,
                            'cantidad': cantidad,
                            'medida': 'cajas (millones) de 18.14kg',
                            'anio': anio_reciente,
                            'framework': "AgenteCustom",
                            'fuente': 'AEBE_BANANOTAS',
                            'archivo_origen': nombre_archivo,
                        })
                    
                    if len(registros) > n_antes:
                        print(f"       ✅ p.{page_num} [{tipo}] {len(registros)-n_antes} registros (año {anio_reciente})")
                        break  # Solo procesar el primer ranking por página

        if not registros:
            print(f"    ⚠️  No se encontraron rankings en el PDF - devolviendo DataFrame vacío")
            return pd.DataFrame(columns=COLUMNAS_VACIAS)

        anio_revista = max(anios_vistos)
        df_final = pd.DataFrame(registros)
        print(f"    ✅ Transformación completada: {len(df_final)} registros | año revista: {anio_revista} | tipos: {sorted(df_final['tipo'].unique())}")
        return df_final

    except Exception as e:
        print(f"    ❌ Error en transformación: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame(columns=COLUMNAS_VACIAS)


print("✅ Función transformar_aebe_bananotas (pdfplumber) cargada.")


# CORRECCIÓN #8: Mejora de transformación ESPAC para múltiples hojas (UNION SIN AMBIGÜEDAD)
def transformar_espac_t13_t26_mejorado_v2(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma archivos ESPAC con múltiples hojas (Tabulados_excel, Series_historicas).
    Detecta automáticamente hojas T13 (banano) y T26 (plátano), aplanando headers fusionados.
    """
    import numpy as np
    import xlrd  # Usar xlrd para leer valores reales, no fórmulas
    
    print(f"  📊 Transformación ESPAC Tabulados/Series Históricas: {nombre_archivo}")
    
    # Extraer año del nombre del archivo
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    try:
        # Leer con pandas/xlrd para obtener nombres de hojas
        xl_file = pd.ExcelFile(local_path)
        nombres_hojas = xl_file.sheet_names
        
        print(f"    📂 Hojas disponibles: {nombres_hojas}")
        
        # Buscar hojas que contengan T13 o T26 en el nombre
        hojas_relevantes = []
        for nombre_hoja in nombres_hojas:
            nombre_upper = nombre_hoja.upper()
            if 'T13' in nombre_upper or 'T26' in nombre_upper or 'BANANO' in nombre_upper or 'PLATANO' in nombre_upper or 'PLÁTANO' in nombre_upper:
                hojas_relevantes.append(nombre_hoja)
        
        print(f"    ✅ Hojas relevantes identificadas: {hojas_relevantes if hojas_relevantes else 'Ninguna, usando hoja activa'}")
        
        dfs_procesados = []
        
        # Si no se encontraron hojas específicas, usar la hoja activa
        if not hojas_relevantes:
            hojas_relevantes = [wb.active.title]
        
        for nombre_hoja in hojas_relevantes:
            print(f"    📄 Procesando hoja: {nombre_hoja}")
            
            try:
                # CORRECCIÓN #9: Leer con pandas directamente (sin openpyxl)
                # Esto lee los VALORES tal como están en las celdas, no códigos internos
                df_temp = pd.read_excel(local_path, sheet_name=nombre_hoja, header=None)
                
                if df_temp.empty:
                    print(f"       ⚠️  Hoja vacía, saltando...")
                    continue
                
                # Convertir todo a string para procesamiento uniforme
                df_temp = df_temp.astype(str).replace({"None":"","nan":"","NaT":"","<NA>":""})
                
                # Buscar fila de header (contiene palabras clave)
                header_idx = 0
                for i in range(min(20, len(df_temp))):
                    fila_text = ' '.join(str(v) for v in df_temp.iloc[i].values).upper()
                    if any(kw in fila_text for kw in ['PROVINCIA', 'SUPERFICIE', 'PRODUCCIÓN', 'PRODUCCION', 'REGION', 'REGIÓN']):
                        header_idx = i
                        print(f"       🎯 Header detectado en fila {header_idx}")
                        break
                
                # Extraer header y datos
                header_vals = list(df_temp.iloc[header_idx].values)
                df_datos = df_temp.iloc[header_idx+1:].copy()
                
                # Normalizar nombres de columnas Y HACER ÚNICOS
                col_names = [normalizar_columna(str(v)) if str(v).strip() else f"col_{i}" for i, v in enumerate(header_vals)]
                # Verificar duplicados y hacer únicos agregando sufijos
                seen = {}
                unique_names = []
                for name in col_names:
                    if name in seen:
                        seen[name] += 1
                        unique_names.append(f"{name}_{seen[name]}")
                    else:
                        seen[name] = 0
                        unique_names.append(name)
                
                df_datos.columns = unique_names
                df_datos = df_datos.reset_index(drop=True)
                
                # Filtrar filas de notas/totales
                palabras_excluir = ['TOTAL', 'REGIÓN', 'REGION', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN', 'OBSERVACION', 
                                   'INSTITUT', 'INEC', 'HTTP', 'WWW', 'ENCUESTA', 'La tabla']
                
                # Identificar columna de provincia/región
                col_provincia = None
                for col in df_datos.columns:
                    if 'provincia' in col.lower() or 'region' in col.lower():
                        col_provincia = col
                        break
                
                if col_provincia:
                    # Convertir a string primero y luego filtrar
                    df_datos = df_datos[df_datos[col_provincia].notna()].copy().reset_index(drop=True)
                    # Filtrar palabras excluidas de forma segura
                    patron = '|'.join(palabras_excluir)
                    mask = df_datos[col_provincia].astype(str).str.upper().str.contains(patron, na=False, case=False, regex=True)
                    df_datos = df_datos[~mask].copy().reset_index(drop=True)
                
                # Agregar información de producto y año
                producto = 'BANANO' if 'T13' in nombre_hoja.upper() or 'BANANO' in nombre_hoja.upper() else 'PLATANO' if 'T26' in nombre_hoja.upper() or 'PLAT' in nombre_hoja.upper() else 'DESCONOCIDO'
                df_datos['producto'] = producto
                
                if anio:
                    df_datos['anio'] = anio
                else:
                    # Intentar extraer año del nombre de la hoja
                    anio_hoja_match = re.search(r'(\d{4})', nombre_hoja)
                    if anio_hoja_match:
                        df_datos['anio'] = int(anio_hoja_match.group(1))
                
                print(f"       ✅ {len(df_datos)} registros procesados de hoja {nombre_hoja}")
                dfs_procesados.append(df_datos)
                
            except Exception as e_hoja:
                print(f"       ❌ Error en hoja {nombre_hoja}: {e_hoja}")
                continue
        
        xl_file.close()
        
        # Combinar todos los DataFrames procesados CON ESQUEMA UNIFICADO
        if dfs_procesados:
            # CORRECCIÓN #13: Mapear columnas POR POSICIÓN en lugar de por nombre
            # La estructura ESPAC es consistente: después del header, las columnas están siempre en el mismo orden
            df_unificados = []
            
            for df_hoja in dfs_procesados:
                n_rows = len(df_hoja)
                df_std = pd.DataFrame(index=range(n_rows))
                
                # CORRECCIÓN #14: Mapeo DIRECTO por nombres de columna conocidos
                # Después de normalizar, buscamos columnas por patrones específicos
                
                print(f"       📊 Columnas disponibles: {list(df_hoja.columns[:15])}")
                
                # Buscar columnas core
                col_provincia = next((c for c in df_hoja.columns if 'provincia' in c or 'region' in c), None)
                col_categoria = next((c for c in df_hoja.columns if 'categor' in c or 'tipo' in c or 'modalidad' in c), None)
                
                # CORRECCIÓN #15: Mapeo por sufijos (superficie_has y superficie_has_1)
                # La normalización crea columnas duplicadas con sufijos _1, _2, etc.
                # superficie_has → plantada, superficie_has_1 → cosechada
                
                # Buscar columnas de superficie (con o sin sufijos)
                superficie_cols = [c for c in df_hoja.columns if 'superficie' in c and 'has' in c]
                col_sup_plantada = superficie_cols[0] if len(superficie_cols) >= 1 else None
                col_sup_cosechada = superficie_cols[1] if len(superficie_cols) >= 2 else None
                
                # Buscar producción y ventas
                col_produccion = None
                col_ventas = None
                for col in df_hoja.columns:
                    col_lower = col.lower()
                    if not col_produccion and 'produccion' in col_lower and 'ventas' not in col_lower:
                        col_produccion = col
                    if not col_ventas and 'ventas' in col_lower:
                        col_ventas = col
                
                print(f"       🎯 Mapeo identificado:")
                print(f"          provincia: {col_provincia}")
                print(f"          categoria: {col_categoria}")
                print(f"          plantada: {col_sup_plantada}")
                print(f"          cosechada: {col_sup_cosechada}")
                print(f"          producción: {col_produccion}")
                print(f"          ventas: {col_ventas}")
                
                # Asignar columnas
                df_std['provincia'] = df_hoja[col_provincia].values if col_provincia else [''] * n_rows
                df_std['categoria'] = df_hoja[col_categoria].values if col_categoria else ['Solo'] * n_rows
                df_std['superficie_plantada_ha'] = df_hoja[col_sup_plantada].values if col_sup_plantada else [0] * n_rows
                df_std['superficie_cosechada_ha'] = df_hoja[col_sup_cosechada].values if col_sup_cosechada else [0] * n_rows
                df_std['produccion_tm'] = df_hoja[col_produccion].values if col_produccion else [0] * n_rows
                df_std['ventas_tm'] = df_hoja[col_ventas].values if col_ventas else [0] * n_rows
                
                df_std['producto'] = df_hoja['producto'].values if 'producto' in df_hoja.columns else ['DESCONOCIDO'] * n_rows
                df_std['anio'] = df_hoja['anio'].values if 'anio' in df_hoja.columns else [0] * n_rows
                
                # CORRECCIÓN #9B: Convertir numéricos con validación de rangos
                for col in ['superficie_plantada_ha', 'superficie_cosechada_ha', 'produccion_tm', 'ventas_tm']:
                    # Limpiar: quitar espacios, reemplazar comas por puntos
                    df_std[col] = df_std[col].astype(str).str.strip().str.replace(',', '.', regex=False)
                    # Quitar puntos solo si hay múltiples (separadores de miles)
                    df_std[col] = df_std[col].apply(lambda x: x.replace('.', '') if x.count('.') > 1 else x)
                    # Convertir a numérico
                    df_std[col] = pd.to_numeric(df_std[col], errors='coerce').fillna(0)
                    
                    # VALIDACIÓN: Valores razonables para Ecuador
                    # Superficie: max 500,000 ha por provincia
                    # Producción/Ventas: max 10,000,000 tm por provincia
                    if 'superficie' in col:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 500000 else x)
                    else:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 10000000 else x)
                
                # Calcular rendimiento
                df_std['rendimiento'] = np.where(
                    df_std['superficie_cosechada_ha'] > 0,
                    (df_std['produccion_tm'] / df_std['superficie_cosechada_ha']).round(2),
                    0
                )
                
                df_unificados.append(df_std)
            
            # Ahora sí, union seguro (todas las hojas tienen el MISMO esquema)
            df_final = pd.concat(df_unificados, ignore_index=True)
            
            # Filtrar filas vacías o inválidas
            df_final = df_final[df_final['provincia'].notna() & (df_final['provincia'].astype(str).str.strip() != '')]
            
            print(f"    ✅ Transformación completada: {len(df_final)} registros totales, {len(df_final.columns)} columnas")
            return df_final
        else:
            print(f"    ⚠️  No se procesaron hojas, retornando DataFrame vacío")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error en transformación ESPAC: {e}")
        print(f"    🔄 Usando lectura estándar como fallback...")
        # Fallback: usar la función leer_archivo estándar
        return leer_archivo(local_path, nombre_archivo)

print("✅ Funciones de transformación especializadas cargadas (AEBE Bananotas y ESPAC Tabulados).")

In [ ]:
def transformar_espac_t13_t26_mejorado_v3(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    CORRECCIÓN #10 FINAL: Mapeo por ÍNDICE de columna fijo (no por nombre).
    Los archivos ESPAC tienen SIEMPRE la misma estructura de columnas:
      - Col 1: Provincia
      - Col 3: Superficie plantada
      - Col 4: Superficie cosechada
      - Col 5: Producción
      - Col 6: Ventas
    """
    import numpy as np
    
    print(f"  📊 Transformación ESPAC Tabulados/Series Históricas (v3 - mapeo por índice): {nombre_archivo}")
    
    # Extraer año del nombre del archivo
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    try:
        # Leer con pandas directamente
        xl_file = pd.ExcelFile(local_path)
        nombres_hojas = xl_file.sheet_names
        
        print(f"    📂 Hojas disponibles: {len(nombres_hojas)} hojas")
        
        # Buscar hojas T13 o T26
        hojas_relevantes = []
        for nombre_hoja in nombres_hojas:
            nombre_upper = nombre_hoja.upper()
            if 'T13' in nombre_upper or 'T26' in nombre_upper:
                hojas_relevantes.append(nombre_hoja)
        
        print(f"    ✅ Hojas relevantes identificadas: {hojas_relevantes if hojas_relevantes else 'Ninguna'}")
        
        if not hojas_relevantes:
            hojas_relevantes = [xl_file.sheet_names[0]]
        
        dfs_procesados = []
        
        for nombre_hoja in hojas_relevantes:
            print(f"    📄 Procesando hoja: {nombre_hoja}")
            
            try:
                # Leer sin header (header=None)
                df_raw = pd.read_excel(local_path, sheet_name=nombre_hoja, header=None)
                
                if df_raw.empty:
                    print(f"       ⚠️  Hoja vacía, saltando...")
                    continue
                
                # Buscar fila de header
                header_idx = 0
                for i in range(min(20, len(df_raw))):
                    fila_text = ' '.join(str(v) for v in df_raw.iloc[i].values if pd.notna(v)).upper()
                    if any(kw in fila_text for kw in ['PROVINCIA', 'SUPERFICIE', 'PRODUCCIÓN', 'PRODUCCION']):
                        header_idx = i
                        print(f"       🎯 Header detectado en fila {header_idx}")
                        break
                
                # Datos empiezan después del header
                df_datos = df_raw.iloc[header_idx+1:].copy().reset_index(drop=True)
                
                # MAPEO POR ÍNDICE DE COLUMNA FIJO (estructura ESPAC estándar)
                # Estructura SIEMPRE es: [0]=índice, [1]=provincia, [2]=categoría?, [3]=sup_plant, [4]=sup_cosech, [5]=prod, [6]=ventas
                # Pero con merge cells, algunas pueden estar en posiciones ligeramente diferentes
                
                # Buscar columna de provincia (primera columna con texto largo)
                col_provincia_idx = None
                for i in range(min(5, df_datos.shape[1])):
                    muestra = df_datos.iloc[:5, i].astype(str)
                    if muestra.str.len().mean() > 5:  # Provincias son texto largo
                        col_provincia_idx = i
                        break
                
                if col_provincia_idx is None:
                    print(f"       ⚠️  No se encontró columna de provincia, saltando...")
                    continue
                
                print(f"       📍 Provincia en col[{col_provincia_idx}]")
                
                # Las columnas numéricas están DESPUÉS de provincia
                # Normalmente: provincia(col 1), luego 4 columnas numéricas
                num_cols_start = col_provincia_idx + 1
                
                # Filtrar filas válidas (que tengan provincia)
                df_datos = df_datos[df_datos.iloc[:, col_provincia_idx].notna()].copy()
                
                # Filtrar palabras excluidas
                palabras_excluir = ['TOTAL', 'REGIÓN', 'REGION', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN']
                patron = '|'.join(palabras_excluir)
                mask = df_datos.iloc[:, col_provincia_idx].astype(str).str.upper().str.contains(patron, na=False, case=False, regex=True)
                df_datos = df_datos[~mask].copy().reset_index(drop=True)
                
                # Crear DataFrame estandarizado
                n_rows = len(df_datos)
                df_std = pd.DataFrame(index=range(n_rows))
                
                df_std['provincia'] = df_datos.iloc[:, col_provincia_idx].values
                df_std['categoria'] = 'Solo'  # No usamos categoría por ahora
                
                # MAPEO NUMÉRICO POR POSICIÓN RELATIVA
                # Después de provincia, las siguientes 4-5 columnas son numéricas
                # Identificar cuáles columnas son numéricas
                cols_numericas = []
                for i in range(num_cols_start, min(num_cols_start + 6, df_datos.shape[1])):
                    muestra = df_datos.iloc[:10, i].astype(str).str.replace(',', '.', regex=False)
                    try:
                        pd.to_numeric(muestra, errors='coerce')
                        es_numerica = muestra.str.match(r'^\d').sum() >= 3  # Al menos 3 valores parecen números
                        if es_numerica:
                            cols_numericas.append(i)
                    except:
                        pass
                
                print(f"       🔢 Columnas numéricas identificadas: {cols_numericas}")
                
                # Asignar en orden: plantada, cosechada, producción, ventas
                if len(cols_numericas) >= 4:
                    df_std['superficie_plantada_ha'] = df_datos.iloc[:, cols_numericas[0]].values
                    df_std['superficie_cosechada_ha'] = df_datos.iloc[:, cols_numericas[1]].values
                    df_std['produccion_tm'] = df_datos.iloc[:, cols_numericas[2]].values
                    df_std['ventas_tm'] = df_datos.iloc[:, cols_numericas[3]].values
                elif len(cols_numericas) >= 3:
                    df_std['superficie_plantada_ha'] = df_datos.iloc[:, cols_numericas[0]].values
                    df_std['superficie_cosechada_ha'] = 0
                    df_std['produccion_tm'] = df_datos.iloc[:, cols_numericas[1]].values
                    df_std['ventas_tm'] = df_datos.iloc[:, cols_numericas[2]].values
                else:
                    print(f"       ⚠️  Insuficientes columnas numéricas ({len(cols_numericas)}), usando ceros")
                    df_std['superficie_plantada_ha'] = 0
                    df_std['superficie_cosechada_ha'] = 0
                    df_std['produccion_tm'] = 0
                    df_std['ventas_tm'] = 0
                
                # Convertir numéricos con validación
                for col in ['superficie_plantada_ha', 'superficie_cosechada_ha', 'produccion_tm', 'ventas_tm']:
                    df_std[col] = df_std[col].astype(str).str.strip().str.replace(',', '.', regex=False)
                    df_std[col] = df_std[col].apply(lambda x: x.replace('.', '') if isinstance(x, str) and x.count('.') > 1 else x)
                    df_std[col] = pd.to_numeric(df_std[col], errors='coerce').fillna(0)
                    
                    # Validación de rangos
                    if 'superficie' in col:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 500000 else x)
                    else:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 10000000 else x)
                
                # Producto y año
                producto = 'BANANO' if 'T13' in nombre_hoja.upper() else 'PLATANO' if 'T26' in nombre_hoja.upper() else 'DESCONOCIDO'
                df_std['producto'] = producto
                df_std['anio'] = anio if anio else 0
                
                # Calcular rendimiento
                df_std['rendimiento'] = np.where(
                    df_std['superficie_cosechada_ha'] > 0,
                    (df_std['produccion_tm'] / df_std['superficie_cosechada_ha']).round(2),
                    0
                )
                
                print(f"       ✅ {len(df_std)} registros procesados de hoja {nombre_hoja}")
                dfs_procesados.append(df_std)
                
            except Exception as e_hoja:
                print(f"       ❌ Error en hoja {nombre_hoja}: {e_hoja}")
                continue
        
        xl_file.close()
        
        if dfs_procesados:
            df_final = pd.concat(dfs_procesados, ignore_index=True)
            df_final = df_final[df_final['provincia'].notna() & (df_final['provincia'].astype(str).str.strip() != '')]
            
            # 🆕 MAPEO DE PROVINCIA A ID (usando tabla dimensional)
            df_final = mapear_provincia_a_id(df_final, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            # 🚫 IMPORTANTE: NO aplicar KNN a provincia_id - los NULL son agregaciones que deben eliminarse
            antes_filtro = len(df_final)
            df_final = df_final.dropna(subset=['provincia_id'])
            despues_filtro = len(df_final)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
            
            # 🚫 KNN DESHABILITADO para ESPAC - provincia_id NULL debe eliminarse, no imputarse
            print(f"    ℹ️  KNN deshabilitado para ESPAC (provincia_id NULL = agregaciones, no errores)")
            
            print(f"    ✅ Transformación completada: {len(df_final)} registros totales, {len(df_final.columns)} columnas")
            return df_final
        else:
            print(f"    ⚠️  No se procesaron hojas, retornando DataFrame vacío")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error en transformación ESPAC: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Activar la versión v3
transformar_espac_t13_t26_mejorado = transformar_espac_t13_t26_mejorado_v3

print("✅ CORRECCIÓN #10 APLICADA: Mapeo por índice de columna (v3)")
print("   - Identifica columna de provincia por contenido de texto")
print("   - Identifica columnas numéricas por patrón de dígitos")
print("   - Asigna en orden: plantada, cosechada, producción, ventas")

## Transformación especializada: SIPA y Uso del Suelo

In [ ]:
def transformar_sipa_temperatura(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos SIPA de temperatura/precipitación.
    Aplica mapeo de provincia a ID.
    """
    print(f"  🌡️  Transformación SIPA Temperatura/Precipitación: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes de SIPA
        columnas_mapeo = {
            'ano': 'anio',
            'mes': 'mes',
            'estacion': 'estacion',
            'provincia': 'provincia',
            'canton': 'canton',
            'precipitacion_mm': 'precipitacion_mm',
            'temperatura_promedio_c': 'temperatura_promedio_c'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # Convertir columnas numéricas - IMPORTANTE: preservar tipo INT
        if 'anio' in df_clean.columns:
            # Convertir a float primero, luego a int para manejar decimales (2000.0 → 2000)
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce')
            # Redondear antes de convertir a int para evitar errores
            df_clean['anio'] = df_clean['anio'].round(0).fillna(0).astype(int)
            # Reemplazar 0 con NaN y eliminar esas filas
            df_clean['anio'] = df_clean['anio'].replace(0, pd.NA)
            # Asegurar que se mantenga como Int64 (nullable int)
            df_clean['anio'] = df_clean['anio'].astype('Int64')
        
        if 'mes' in df_clean.columns:
            # Convertir a int, manejando valores vacíos
            df_clean['mes'] = pd.to_numeric(df_clean['mes'], errors='coerce')
            df_clean['mes'] = df_clean['mes'].fillna(0).astype(int)
            # Si todos son 0, intentar extraer mes de otra columna si existe
            if (df_clean['mes'] == 0).all() and 'anio' in df_clean.columns:
                # Si no hay mes, agregar columna con 0 para indicar dato anual
                print(f"    ℹ️  Columna 'mes' vacía, usando 0 (dato anual)")
        elif 'anio' in df_clean.columns:
            # Si no existe columna mes, crearla con 0
            df_clean['mes'] = 0
            print(f"    ℹ️  Columna 'mes' no existe, creada con 0 (dato anual)")
        
        if 'precipitacion_mm' in df_clean.columns:
            df_clean['precipitacion_mm'] = pd.to_numeric(df_clean['precipitacion_mm'], errors='coerce')
        
        if 'temperatura_promedio_c' in df_clean.columns:
            df_clean['temperatura_promedio_c'] = pd.to_numeric(df_clean['temperatura_promedio_c'], errors='coerce')
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧹 Filtradas {antes - despues} filas con anio nulo")
        
        # 🎯 MAPEO DE PROVINCIA A ID (usando tabla dimensional)
        if 'provincia' in df_clean.columns:
            df_clean = mapear_provincia_a_id(df_clean, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            antes_filtro = len(df_clean)
            df_clean = df_clean.dropna(subset=['provincia_id'])
            despues_filtro = len(df_clean)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
                print(f"    ✅ {despues_filtro} registros con provincia válida")
        
        # ✅ APLICAR KNN A COLUMNAS NUMÉRICAS EN SIPA
        print(f"    🔧 Aplicando KNN a columnas numéricas con NULL...")
        
        # Identificar columnas numéricas con NULL (excluir IDs)
        columnas_numericas_con_null = []
        for col in df_clean.columns:
            if col in ['provincia_id', 'anio', 'mes']:  # No aplicar KNN a IDs y fechas
                continue
            try:
                col_numeric = pd.to_numeric(df_clean[col], errors='coerce')
                nulls = col_numeric.isna().sum()
                if nulls > 0 and nulls < len(df_clean):  # Tiene NULL pero no todos
                    columnas_numericas_con_null.append(col)
                    print(f"       {col}: {nulls} NULL ({nulls/len(df_clean)*100:.1f}%)")
            except:
                pass
        
        if columnas_numericas_con_null and len(df_clean) >= 5:
            try:
                from sklearn.impute import KNNImputer
                
                # Preparar datos para KNN
                features_knn = columnas_numericas_con_null.copy()
                if 'anio' in df_clean.columns:
                    features_knn = ['anio'] + features_knn
                if 'mes' in df_clean.columns and df_clean['mes'].nunique() > 1:
                    features_knn.insert(1, 'mes')
                
                # Convertir a numérico
                df_knn = df_clean[features_knn].copy()
                for col in df_knn.columns:
                    df_knn[col] = pd.to_numeric(df_knn[col], errors='coerce')
                
                # Aplicar KNN
                imputer = KNNImputer(n_neighbors=min(5, len(df_clean)), weights='distance')
                df_imputed = imputer.fit_transform(df_knn)
                
                # Reemplazar valores en columnas originales (sin año/mes)
                start_idx = 0
                if 'anio' in features_knn:
                    start_idx += 1
                if 'mes' in features_knn:
                    start_idx += 1
                
                for i, col in enumerate(columnas_numericas_con_null):
                    df_clean[col] = df_imputed[:, start_idx + i]
                
                print(f"    ✅ KNN aplicado a {len(columnas_numericas_con_null)} columnas")
            except Exception as e:
                print(f"    ⚠️  No se pudo aplicar KNN: {e}")
        else:
            print(f"    ℹ️  KNN omitido (no hay suficientes datos o columnas)")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        # Eliminar columna 'mes': dato auxiliar interno, no se exporta
        df_clean = df_clean.drop(columns=['mes'], errors='ignore')
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación SIPA: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación SIPA Temperatura cargada (con mapeo de provincia_id y KNN)")

In [ ]:
def transformar_uso_del_suelo(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos de Uso del Suelo.
    La columna 'region_y_provincia' puede contener nombres de provincias O regiones.
    Solo mapeamos las provincias a provincia_id. Las regiones quedan sin mapear (NULL).
    """
    print(f"  🌾 Transformación Uso del Suelo: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes
        columnas_mapeo = {
            'ano': 'anio',
            'region_y_provincia': 'region_y_provincia',
            'categoria_de_uso_del_suelo': 'categoria_de_uso_del_suelo',
            'superficie_ha': 'superficie_ha'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # Convertir columnas numéricas
        if 'anio' in df_clean.columns:
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce').astype('Int64')
        
        if 'superficie_ha' in df_clean.columns:
            df_clean['superficie_ha'] = pd.to_numeric(df_clean['superficie_ha'], errors='coerce')
        
        # 🎯 SEPARAR region_y_provincia EN SOLO provincia
        # La columna puede tener nombres de provincia directos ("Azuay", "Guayas")
        # o nombres de región ("Centro-Suroriente", "Costa", etc.)
        # Solo mapeamos provincias, las regiones quedan como NULL y se ELIMINAN
        if 'region_y_provincia' in df_clean.columns:
            # Renombrar temporalmente para el mapeo
            df_clean = df_clean.rename(columns={'region_y_provincia': 'provincia'})
            
            print(f"    🗺️  Intentando mapear valores de región_y_provincia a provincia_id...")
            
            # Aplicar mapeo - los valores que NO son provincias quedarán NULL
            df_clean = mapear_provincia_a_id(df_clean, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            antes_filtro = len(df_clean)
            df_clean = df_clean.dropna(subset=['provincia_id'])
            despues_filtro = len(df_clean)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
                print(f"    ✅ {despues_filtro} registros con provincia válida")
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧹 Filtradas {antes - despues} filas con anio nulo")
        
        # ✅ APLICAR KNN A COLUMNAS NUMÉRICAS EN USO DEL SUELO
        print(f"    🔧 Aplicando KNN a columnas numéricas con NULL...")
        
        # Identificar columnas numéricas con NULL (excluir IDs)
        columnas_numericas_con_null = []
        for col in df_clean.columns:
            if col in ['provincia_id', 'anio']:  # No aplicar KNN a IDs y fechas
                continue
            try:
                col_numeric = pd.to_numeric(df_clean[col], errors='coerce')
                nulls = col_numeric.isna().sum()
                if nulls > 0 and nulls < len(df_clean):  # Tiene NULL pero no todos
                    columnas_numericas_con_null.append(col)
                    print(f"       {col}: {nulls} NULL ({nulls/len(df_clean)*100:.1f}%)")
            except:
                pass
        
        if columnas_numericas_con_null and len(df_clean) >= 5:
            try:
                from sklearn.impute import KNNImputer
                
                # Preparar datos para KNN
                features_knn = columnas_numericas_con_null.copy()
                if 'anio' in df_clean.columns:
                    features_knn = ['anio'] + features_knn
                if 'provincia_id' in df_clean.columns:
                    features_knn.insert(0 if 'anio' not in features_knn else 1, 'provincia_id')
                
                # Convertir a numérico
                df_knn = df_clean[features_knn].copy()
                for col in df_knn.columns:
                    df_knn[col] = pd.to_numeric(df_knn[col], errors='coerce')
                
                # Aplicar KNN
                imputer = KNNImputer(n_neighbors=min(5, len(df_clean)), weights='distance')
                df_imputed = imputer.fit_transform(df_knn)
                
                # Reemplazar valores en columnas originales (sin provincia_id/año)
                start_idx = 0
                if 'provincia_id' in features_knn:
                    start_idx += 1
                if 'anio' in features_knn:
                    start_idx += 1
                
                for i, col in enumerate(columnas_numericas_con_null):
                    df_clean[col] = df_imputed[:, start_idx + i]
                
                print(f"    ✅ KNN aplicado a {len(columnas_numericas_con_null)} columnas")
            except Exception as e:
                print(f"    ⚠️  No se pudo aplicar KNN: {e}")
        else:
            print(f"    ℹ️  KNN omitido (no hay suficientes datos o columnas)")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación Uso del Suelo: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación Uso del Suelo cargada (con mapeo de provincia_id y KNN)")

## Funciones auxiliares para AEBE (limpieza de números y porcentajes)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES PARA AEBE (limpieza de números y porcentajes)
# ══════════════════════════════════════════════════════════════════════

def limpiar_numero(valor) -> float:
    """
    Extrae un número flotante de un string, manejando:
    - Separadores de miles (comas, puntos)
    - Decimales
    - Valores NULL/vacíos
    
    Ejemplos:
        "17,084.5" → 17084.5
        "17.084,5" → 17084.5
        "25.79%" → 25.79
        "" → None
    """
    if pd.isna(valor) or valor == '' or valor is None:
        return None
    
    try:
        # Convertir a string y limpiar
        s = str(valor).strip()
        
        # Eliminar símbolos no numéricos (excepto . , - +)
        s = re.sub(r'[^\d.,-]', '', s)
        
        if not s or s in ['.', ',', '-']:
            return None
        
        # Detectar formato: si tiene coma después de punto, es formato europeo
        if ',' in s and '.' in s:
            if s.rindex(',') > s.rindex('.'):
                # Formato europeo: 1.234,56
                s = s.replace('.', '').replace(',', '.')
            else:
                # Formato americano: 1,234.56
                s = s.replace(',', '')
        elif ',' in s:
            # Solo comas: puede ser separador de miles o decimal
            # Si hay más de una coma, es separador de miles
            if s.count(',') > 1:
                s = s.replace(',', '')
            else:
                # Una sola coma: puede ser decimal (europeo) o miles (americano)
                # Asumimos decimal si hay 1-2 dígitos después
                partes = s.split(',')
                if len(partes) == 2 and len(partes[1]) <= 2:
                    s = s.replace(',', '.')
                else:
                    s = s.replace(',', '')
        
        return float(s)
    
    except (ValueError, AttributeError):
        return None


def limpiar_porcentaje(valor) -> float:
    """
    Extrae un porcentaje y lo convierte a decimal.
    
    Ejemplos:
        "25.79%" → 25.79
        "4.39" → 4.39
        "25,79%" → 25.79
    """
    if pd.isna(valor) or valor == '' or valor is None:
        return None
    
    try:
        s = str(valor).strip()
        # Eliminar símbolo de porcentaje
        s = s.replace('%', '').strip()
        return limpiar_numero(s)
    
    except:
        return None

print("✅ Funciones auxiliares AEBE cargadas: limpiar_numero(), limpiar_porcentaje()")

## Transformación especializada: FAOSTAT

In [ ]:
def transformar_faostat(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos FAOSTAT JSON.
    Normaliza columnas y selecciona campos relevantes.
    """
    print(f"  🌍 Transformación FAOSTAT: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes de FAOSTAT
        columnas_mapeo = {
            'area': 'pais',
            'area_code': 'pais_codigo',
            'item': 'producto',
            'item_code': 'producto_codigo',
            'element': 'elemento',
            'element_code': 'elemento_codigo',
            'year': 'anio',
            'unit': 'unidad',
            'value': 'valor',
            'flag': 'bandera'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # CORRECCIÓN #16: Eliminar columnas redundantes
        # year_code es idéntico a anio después del mapeo
        columnas_redundantes = ['year_code']
        for col_redundante in columnas_redundantes:
            if col_redundante in df_clean.columns:
                df_clean = df_clean.drop(columns=[col_redundante])
                print(f"    🗑️  Eliminada columna redundante: {col_redundante}")
        
        # Convertir columnas numéricas
        if 'anio' in df_clean.columns:
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce').astype('Int64')
        
        if 'valor' in df_clean.columns:
            df_clean['valor'] = pd.to_numeric(df_clean['valor'], errors='coerce')
        
        if 'pais_codigo' in df_clean.columns:
            df_clean['pais_codigo'] = pd.to_numeric(df_clean['pais_codigo'], errors='coerce').astype('Int64')
        
        if 'producto_codigo' in df_clean.columns:
            df_clean['producto_codigo'] = pd.to_numeric(df_clean['producto_codigo'], errors='coerce').astype('Int64')
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns and 'valor' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio', 'valor'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧼 Filtradas {antes - despues} filas con anio/valor nulos")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación FAOSTAT: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación FAOSTAT cargada")

## Estado del agente ETL y pasos base (detección, lectura, mapeo por reglas, finalización)

In [ ]:
# ── DEFINICIÓN DEL ESTADO ──────────────────────────────────────────────────
class ETLState(TypedDict):
    # ── Input ─────────────────────────────────────────────────────────────
    file_name:              str
    local_path:             str
    # ── Paso 1: Detección ─────────────────────────────────────────────────
    fuente:                 str
    inicio_ts:              str          # timestamp ISO inicio general
    # ── Paso 2: Lectura ───────────────────────────────────────────────────
    df_columnas:            List[str]
    total_filas:            int
    ts_fin_lectura:         str          # timestamp ISO fin lectura
    error_lectura:          Optional[str]
    # ── Paso 3: Mapeo Híbrido (LLM + Reglas) ──────────────────────────────────────
    tabla_destino:          str
    cols_double:            List[str]
    cols_integer:           List[str]
    llm_response:           str
    llamadas_api:           int
    error_mapeo:            Optional[str]
    # ── Paso 4: Transformación ────────────────────────────────────────────
    registros_validos:      int
    registros_duplicados:   int
    ts_fin_transformacion:  str          # timestamp ISO fin transformación
    error_transform:        Optional[str]
    temp_view_name:         str          # nombre único temp view Spark
    # ── Paso 5: Carga ─────────────────────────────────────────────────────
    tabla_completa:         str
    ts_fin_carga:           str          # timestamp ISO fin carga
    error_carga:            Optional[str]
    # ── Paso 6: Finalización ──────────────────────────────────────────────
    tiempo_segundos:        float
    status:                 str
    error_final:            Optional[str]

# ── PASO 1: DETECCIÓN ─────────────────────────────────────────────────────
def paso_deteccion(state: ETLState) -> dict:
    """
    Primer paso del agente. Identifica el organismo de origen del archivo
    y registra el timestamp de inicio del procesamiento.
    """
    print(f"\n{'='*55}")
    print(f"[PASO 1 — DETECCIÓN] {state['file_name']}")
    fuente = identificar_fuente(state["file_name"])
    print(f"  Fuente: {fuente}")
    return {"fuente": fuente, "inicio_ts": datetime.now().isoformat(), "llamadas_api": 0}

# ── PASO 2: LECTURA ───────────────────────────────────────────────────────
def paso_lectura(state: ETLState) -> dict:
    """
    Lee el archivo físico del volumen y detecta automáticamente el header.
    Registra ts_fin_lectura como referencia de tiempo.
    """
    print(f"[PASO 2 — LECTURA]")
    try:
        df = leer_archivo(state["local_path"], state["file_name"])
        n_filas, n_cols = df.shape
        print(f"  {n_filas} filas × {n_cols} columnas")
        print(f"  Primeras columnas: {list(df.columns)[:5]}")
        if n_filas == 0:
            return {"error_lectura":"Archivo vacío","total_filas":0,"df_columnas":[],
                    "ts_fin_lectura": datetime.now().isoformat()}
        return {"df_columnas":list(df.columns), "total_filas":n_filas, "error_lectura":None,
                "ts_fin_lectura": datetime.now().isoformat()}
    except Exception as e:
        print(f"  ❌ {e}")
        return {"error_lectura":str(e),"total_filas":0,"df_columnas":[],
                "ts_fin_lectura": datetime.now().isoformat()}

# ── PASO 3: MAPEO SEMÁNTICO CON LLM + FALLBACK ──────────────────────────
def _mapeo_basado_en_reglas(file_name: str, columnas: List[str]) -> dict:
    """
    Fallback basado en reglas cuando el LLM falla.
    Usa el nombre del archivo y columnas para determinar la tabla destino.
    """
    fn = file_name.upper()
    cols_text = " ".join(columnas).lower()
    
    # Reglas por nombre de archivo
    # PRIORIDAD MÁXIMA: ESPAC Tabulados y Series históricas → TABLA UNIFICADA banano/plátano
    # (debe estar ANTES de otras reglas para evitar falsos positivos)
    if "TABULADOS_EXCEL" in fn or "TABULADOS" in fn or "SERIES_HISTORICAS" in fn or "SERIES_HIST" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    # ESPAC T13 (banano) y T26 (plátano) → TABLA UNIFICADA
    if "T13" in fn or "T26" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    if "USO_DEL_SUELO" in fn or "uso" in cols_text and "suelo" in cols_text:
        return {"tabla":"espac_uso_del_suelo", "conf":0.85, "double":["superficie_ha"], "int":["ano"]}
    if "TEMPERATURA" in fn or ("temperatura" in cols_text and "precipitacion" in cols_text):
        return {"tabla":"sipa_temperatura_precipitacion", "conf":0.95, "double":["precipitacion_mm","temperatura_promedio_c"], "int":["ano"]}
    
    # FAOSTAT bananas y plantains → TABLA UNIFICADA
    if "FAOSTAT" in fn and ("BANANAS" in fn or "PLANTAINS" in fn):
        return {"tabla":"faostat_produccion_banano_platano", "conf":0.95, "double":["value"], "int":["year","area_code","item_code"]}
    
    if "AEBE" in fn or "BANANOTAS" in fn:
        return {"tabla":"aebe_exportaciones_regiones", "conf":0.95, "double":["cantidad"], "int":["anio"]}
    
    # Fallback genérico
    return {"tabla":"tabla_temporal", "conf":0.3, "double":[], "int":[]}

# ── FUNCIÓN AUXILIAR: TRANSFORMACIÓN ESPECIALIZADA T13/T26 ───────────────
def transformar_espac_t13_t26(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma archivos T13 (banano) y T26 (plátano) de ESPAC al formato estandarizado.
    Pasos: Elimina sub-headers, renombra columnas, filtra provincias y notas, convierte numéricos, extrae año.
    """
    import numpy as np
    print(f"  📋 Transformación específica T13/T26 para: {nombre_archivo}")
    
    # Extraer año del nombre del archivo con regex flexible
    # Captura: _2021_, 2021.csv, ESPAC2021, T13_2022, etc.
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    df = df_pandas.iloc[1:].copy()
    # Detección robusta de columna provincia (no asignación posicional fija)
    cols_orig = list(df.columns)
    col_provincia = next((c for c in cols_orig if 'PROVINCIA' in str(c).upper()), None)
    if col_provincia is None:
        col_provincia = cols_orig[0]
        print(f"  ⚠️ Columna PROVINCIA no encontrada, usando '{col_provincia}' como fallback")
    expected_cols = ['PROVINCIA','CATEGORIA','SUPERFICIE_PLANTADA_HA','SUPERFICIE_COSECHADA_HA','PRODUCCION_TM','VENTAS_TM']
    rename_map = {cols_orig[idx]: name for idx, name in enumerate(expected_cols) if idx < len(cols_orig)}
    df = df.rename(columns=rename_map)
    for col_needed in expected_cols:
        if col_needed not in df.columns:
            df[col_needed] = None
    
    # Filtrar filas que NO son datos reales (provincias)
    palabras_excluir = ['TOTAL', 'REGIÓN', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN', 'OBSERVACION', 
                        'La tabla', 'INSTITUT', 'INEC', 'HTTP', 'WWW', 'ENCUESTA']
    
    df = df[
        df['PROVINCIA'].notna() &
        (~df['PROVINCIA'].astype(str).str.upper().str.contains('|'.join(palabras_excluir), na=False, case=False, regex=True))
    ].copy()
    
    print(f"    🗑️  Filtradas filas con notas/totales/fuentes")
    print(f"    → {len(df)} provincias válidas después del filtrado")
    
    # Convertir columnas numéricas
    for col in ['SUPERFICIE_PLANTADA_HA', 'SUPERFICIE_COSECHADA_HA', 'PRODUCCION_TM', 'VENTAS_TM']:
        df[col] = df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Agregar columna de año
    df['ANIO'] = anio if anio else 0
    
    # Agregar columnas calculadas
    df['PRODUCTO'] = 'BANANO' if 'T13' in nombre_archivo.upper() else 'PLATANO' if 'T26' in nombre_archivo.upper() else 'DESCONOCIDO'
    df['RENDIMIENTO'] = np.where(df['SUPERFICIE_COSECHADA_HA'] > 0, (df['PRODUCCION_TM'] / df['SUPERFICIE_COSECHADA_HA']).round(2), 0)
    
    # Limpiar strings
    df['PROVINCIA'] = df['PROVINCIA'].astype(str).str.strip()
    df['CATEGORIA'] = df['CATEGORIA'].fillna('Solo').astype(str).str.strip()
    
    df.reset_index(drop=True, inplace=True)
    print(f"    ✅ Transformación completada: {len(df)} registros, {len(df.columns)} columnas (incluye ANIO)")
    return df


# ── PASO 6: FINALIZACIÓN ─────────────────────────────────────────────────
def paso_finalizar(state: ETLState) -> dict:
    """
    Calcula el tiempo total del procesamiento de este archivo y determina
    el status final (PROCESADO / ERROR), sin registrar métricas en Delta.
    """
    fin    = datetime.now()
    inicio = datetime.fromisoformat(state["inicio_ts"])
    tiempo = (fin - inicio).total_seconds()

    hay_error = any([state.get("error_lectura"), state.get("error_mapeo"),
                     state.get("error_transform"), state.get("error_carga")])
    status = "ERROR" if hay_error else "PROCESADO"

    error_msg = " | ".join(filter(None, [
        state.get("error_lectura"), state.get("error_mapeo"),
        state.get("error_transform"), state.get("error_carga"),
    ])) or None

    print(f"[PASO 6 — FINALIZACIÓN] {state['file_name']} → {status} ({tiempo:.1f}s)")

    return {"tiempo_segundos": tiempo, "status": status, "error_final": error_msg}

# ── PASO ERROR ────────────────────────────────────────────────────────────
def paso_error(state: ETLState) -> dict:
    """
    Paso de manejo centralizado de errores.
    El agente lo invoca automáticamente cuando cualquier paso anterior falla,
    en lugar de continuar con el resto de la cadena.
        """
    print(f"[PASO ERROR] — {state.get('file_name','?')}")
    errores = " | ".join(filter(None,[
        state.get("error_lectura"),state.get("error_mapeo"),
        state.get("error_transform"),state.get("error_carga"),
    ]))
    print(f"  Causa: {errores or 'desconocida'}")
    return {"status":"ERROR","error_final":errores or "Error desconocido"}

print("✅ Estado ETLState y pasos base del agente (detección, lectura, métricas, error) definidos.")

## Mapeo híbrido (LLM + reglas)

In [ ]:
def paso_mapeo(state: ETLState) -> dict:
    """
    Mapeo híbrido de la tabla destino: intenta el LLM primero y usa reglas
    por nombre/columnas como fallback si falla. Para AEBE siempre usa
    reglas (el nombre del archivo ya es conocido). Siempre elimina
    extensiones de archivo si el LLM las incluye por error.
    """
    print(f"[PASO 3 — MAPEO HÍBRIDO] columnas: {state['df_columnas'][:4]}...")
    llamadas = state.get("llamadas_api", 0)
    fn_upper = state["file_name"].upper()

    if "AEBE" in fn_upper or "BANANOTAS" in fn_upper:
        fallback = _mapeo_basado_en_reglas(state["file_name"], state["df_columnas"])
        print(f"  🛡️  Archivo AEBE — mapeo forzado por reglas: '{fallback['tabla']}'")
        return {"tabla_destino":fallback["tabla"],
                "cols_double":fallback["double"],
                "cols_integer":fallback["int"],
                "llm_response":json.dumps({"metodo":"regla_forzada_aebe","confianza":fallback["conf"]},ensure_ascii=False),
                "llamadas_api":llamadas,
                "error_mapeo":None}

    try:
        print(f"  LLM — intento 1/1...")
        
        # Prompt mejorado con advertencia explícita
        prompt_mejorado = f"""Identifica la tabla destino para el archivo: {state['file_name']}
Columnas encontradas: {', '.join(state['df_columnas'][:8])}

Tablas válidas (elige EXACTAMENTE una):
- espac_banano_platano_provincia (T13/T26, Tabulados, Series Históricas - banano/plátano por provincia UNIFICADO)
- espac_uso_del_suelo (uso del suelo)
- sipa_temperatura_precipitacion (clima)
- faostat_produccion_banano_platano (FAO bananas/plantains UNIFICADO)
- aebe_exportaciones_regiones (AEBE Bananotas - rankings de exportaciones por región/país)

⚠️ SI el archivo tiene "Tabulados" o "Series" en el nombre → SIEMPRE usar espac_banano_platano_provincia
⚠️ SI el archivo tiene "AEBE" o "BANANOTAS" en el nombre → SIEMPRE usar aebe_exportaciones_regiones

⚠️ IMPORTANTE: La tabla_destino debe ser EXACTAMENTE uno de los nombres arriba.
⚠️ NO incluyas extensiones (.xlsx, .csv, .xls) en el nombre.

Responde SOLO con JSON válido (sin markdown):
{{"tabla_destino":"nombre_sin_extension","confianza":0.9,"columnas_double":[],"columnas_integer":[]}}"""
        
        resp = llm.invoke([HumanMessage(content=prompt_mejorado)])
        llamadas += 1
        
        if resp.content and resp.content.strip():
            texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
            start = texto.find("{")
            end = texto.rfind("}")
            if start != -1 and end != -1:
                texto = texto[start:end+1]
                mapeo = json.loads(texto)
                tabla = mapeo.get("tabla_destino","").lower().replace(" ","_")
                
                # ✅ VALIDACIÓN: Eliminar extensiones si el LLM las incluyó
                for ext in [".xlsx", ".xls", ".csv", ".json"]:
                    if tabla.endswith(ext):
                        print(f"  ⚠️ LLM incluyó extensión '{ext}' — removiendo...")
                        tabla = tabla[:-len(ext)]
                
                if tabla and tabla != "tabla_temporal":
                    print(f"  ✅ LLM: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                    return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                            "cols_integer":mapeo.get("columnas_integer",[]),
                            "llm_response":json.dumps(mapeo,ensure_ascii=False),
                            "llamadas_api":llamadas, "error_mapeo":None}
    except Exception as e:
        print(f"  ⚠️ LLM falló: {str(e)[:50]}...")
    
    # Fallback a reglas
    print(f"  🔧 Usando mapeo basado en reglas...")
    fallback = _mapeo_basado_en_reglas(state["file_name"], state["df_columnas"])
    print(f"  ✅ Regla: '{fallback['tabla']}' (confianza: {fallback['conf']})")
    
    return {"tabla_destino":fallback["tabla"], 
            "cols_double":fallback["double"],
            "cols_integer":fallback["int"],
            "llm_response":json.dumps({"metodo":"fallback_reglas","confianza":fallback["conf"]},ensure_ascii=False),
            "llamadas_api":llamadas, 
            "error_mapeo":None}

print("✅ Paso de mapeo definido (LLM con fallback a reglas).")

## Transformación y carga a Delta

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PASOS DE TRANSFORMACIÓN Y CARGA DEL AGENTE ETL
# ══════════════════════════════════════════════════════════════════════════

def paso_transformacion(state: ETLState) -> dict:
    """
    Aplica la transformación especializada según la fuente detectada,
    castea tipos, limpia valores y elimina duplicados sobre una vista
    temporal de Spark lista para cargarse a Delta.
    """
    print(f"[PASO 4 — TRANSFORMACIÓN]")
    ts_ini_t = datetime.now()
    try:
        nombre_upper = state["file_name"].upper()
        
        # ── TRANSFORMACIONES ESPECIALIZADAS POR FUENTE ──
        if "AEBE" in nombre_upper or "BANANOTAS" in nombre_upper:
            print(f"  🎯 Detectado archivo AEBE Bananotas — Aplicando transformación especializada (pdfplumber)")
            df_pandas = transformar_aebe_bananotas(state["local_path"], state["file_name"])
        
        elif "TABULADOS_EXCEL" in nombre_upper or "SERIES_HISTORICAS" in nombre_upper or "SERIES_HIST" in nombre_upper:
            print(f"  🎯 Detectado ESPAC Tabulados/Series Históricas — Aplicando transformación de múltiples hojas")
            df_pandas = transformar_espac_t13_t26_mejorado(state["local_path"], state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        elif "T13" in nombre_upper or "T26" in nombre_upper:
            print(f"  🎯 Detectado archivo T13/T26 — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_espac_t13_t26(df_pandas, state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        elif "FAOSTAT" in nombre_upper:
            print(f"  🎯 Detectado archivo FAOSTAT — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_faostat(df_pandas, state["file_name"])
        
        elif "TEMPERATURA" in nombre_upper or ("SIPA" in nombre_upper and "USO" not in nombre_upper):
            print(f"  🎯 Detectado SIPA Temperatura — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_sipa_temperatura(df_pandas, state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        elif "USO_DEL_SUELO" in nombre_upper or "USO" in nombre_upper:
            print(f"  🎯 Detectado Uso del Suelo — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_uso_del_suelo(df_pandas, state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id (totales regionales)")
        
        else:
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
        
        df_spark  = spark.createDataFrame(df_pandas)
        df_spark  = castear_columnas(df_spark, state["cols_double"], state["cols_integer"])
        df_spark  = (df_spark
            .withColumn("pipeline_source_file", F.lit(state["file_name"]))
            .withColumn("pipeline_framework",   F.lit("AgenteCustom"))
            .withColumn("pipeline_load_ts",     F.current_timestamp()))

        # Limpiar columnas PERO preservar tipos INT/BIGINT
        cols_to_clean = df_spark.columns
        clean_exprs = {}
        
        # Identificar columnas que deben mantenerse como enteros
        columnas_enteras = ['anio', 'ano', 'year', 'mes', 'provincia_id', 'canton_id', 'pais_codigo', 'producto_codigo']
        
        for c in cols_to_clean:
            # Obtener el tipo actual de la columna
            col_tipo = dict(df_spark.dtypes)[c]
            
            # Si es una columna que debe ser entera Y ya es IntegerType/LongType, preservarla
            if c.lower() in columnas_enteras and col_tipo in ['int', 'bigint', 'integer', 'long']:
                # NO convertir a string, solo limpiar nulls/ceros invalidos
                clean_exprs[c] = when(
                    col(c).isNull() | (col(c) == 0),
                    None).otherwise(col(c))
            else:
                # Para otras columnas, convertir a string y limpiar
                clean_exprs[c] = when(
                    trim(col(c).cast("string")).isin("","null","NULL","None","nan","NaN",".","-","0"),
                    None).otherwise(trim(col(c).cast("string")))
        
        df_spark = df_spark.withColumns(clean_exprs)
        df_spark = df_spark.na.drop(how="all")

        cols_anio = [c for c in df_spark.columns if c.lower() in ['anio', 'ano', 'year', 'fecha']]
        if not cols_anio:
            anio_match = re.search(r'(\d{4})', state["file_name"])
            if anio_match:
                anio_extraido = int(anio_match.group(1))
                df_spark = df_spark.withColumn("anio", F.lit(anio_extraido))
                print(f"  📅 Año extraído del nombre: {anio_extraido}")
        
        if "aebe" in state.get("tabla_destino", "").lower():
            total_rows = df_spark.count()
            if total_rows > 0:
                meta_cols_preserve = {'_input_file_name', '_rescued_data', '_metadata', 'anio', 'pipeline_source_file', 'pipeline_load_ts', 'pipeline_framework'}
                cols_to_drop = []
                for col_name in df_spark.columns:
                    if col_name not in meta_cols_preserve:
                        null_count = df_spark.filter(col(col_name).isNull() | (trim(col(col_name)) == "")).count()
                        null_pct = (null_count / total_rows) * 100
                        if null_pct > 95:
                            cols_to_drop.append(col_name)
                
                if cols_to_drop:
                    df_spark = df_spark.drop(*cols_to_drop)
                    print(f"  🗑️  Eliminadas {len(cols_to_drop)} columnas vacías (>95% nulos)")
        
        print(f"  ✓ Filtro de producto deshabilitado (archivos pre-filtrados por fuente)")

        meta_cols = {"pipeline_source_file","pipeline_load_ts","pipeline_framework"}
        all_columns = df_spark.columns
        real_cols = [c for c in all_columns if c not in meta_cols]
        
        antes = df_spark.count()
        df_spark = df_spark.dropDuplicates(subset=real_cols)
        despues = df_spark.count()
        duplicados = antes - despues

        temp_view_name = f"agente_etl_temp_{state['file_name'].replace('.','_').replace('-','_')}"
        df_spark.createOrReplaceTempView(temp_view_name)

        ts_fin_t = datetime.now()
        tiempo_t = (ts_fin_t - ts_ini_t).total_seconds()

        print(f"  Registros: {antes} → {despues} (−{duplicados} duplicados)  Tiempo={tiempo_t:.1f}s")
        return {"registros_validos":despues, "registros_duplicados":duplicados,
                "ts_fin_transformacion": ts_fin_t.isoformat(), "error_transform":None,
                "temp_view_name": temp_view_name}

    except Exception as e:
        print(f"  ❌ {e}")
        return {"registros_validos":0,"registros_duplicados":0,
                "ts_fin_transformacion": datetime.now().isoformat(), "error_transform":str(e)}


def paso_carga(state: ETLState) -> dict:
    """
    Escribe el DataFrame transformado (vista temporal) en la tabla Delta
    destino dentro de bd_banano_ec, usando append + mergeSchema.
    """
    tabla_completa = f"{DB_NAME}.{state['tabla_destino']}"
    print(f"[PASO 5 — CARGA] → {tabla_completa}")
    ts_ini_c = datetime.now()
    try:
        temp_view_name = state.get("temp_view_name", "agente_etl_temp")
        df_spark = spark.table(temp_view_name)
        
        registros_a_insertar = df_spark.count()

        if spark.catalog.tableExists(tabla_completa):
            schema_dest = spark.table(tabla_completa).schema
            df_cols = df_spark.columns
            cast_exprs = {}
            for campo in schema_dest.fields:
                if campo.name in df_cols:
                    cast_exprs[campo.name] = col(campo.name).cast(campo.dataType)
            if cast_exprs:
                df_spark = df_spark.withColumns(cast_exprs)

        df_spark.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(tabla_completa)
        ts_fin_c  = datetime.now()
        tiempo_c  = (ts_fin_c - ts_ini_c).total_seconds()

        print(f"  ✅ {registros_a_insertar} registros en {tabla_completa} ({tiempo_c:.1f}s)")

        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":None}

    except Exception as e:
        ts_fin_c = datetime.now()
        print(f"  ❌ {e}")
        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":str(e)}


print("✅ Pasos de transformación y carga del agente definidos.")

## Lógica de enrutamiento entre pasos

In [ ]:
def ruta_post_lectura(state):
    if state.get("error_lectura"):
        print("  → Ruta: ERROR (lectura fallida)")
        return "error"
    return "mapeo"

def ruta_post_mapeo(state):
    if state.get("error_mapeo") and state.get("tabla_destino") == "tabla_fallback":
        print("  → Ruta: ERROR (mapeo fallido)")
        return "error"
    return "transformacion"

def ruta_post_transformacion(state):
    if state.get("error_transform"):
        print("  → Ruta: ERROR (transformación fallida)")
        return "error"
    return "carga"

def ruta_post_carga(state):
    if state.get("error_carga"):
        print("  → Ruta: ERROR (carga fallida)")
        return "error"
    return "metricas"

print("✅ 4 condiciones de enrutamiento definidas.")

## Ensamblado del agente ETL

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FUNCIONES WRAPPER PARA EJECUTAR LOS AGENTES
# ══════════════════════════════════════════════════════════════════════════

def ejecutar_agente_etl(estado_inicial: dict):
    """
    Ejecuta el agente ETL personalizado.
    """
    return agente_etl.ejecutar(estado_inicial)

print("✅ Funciones wrapper de ejecución definidas")

In [ ]:
class AgenteETL:
    """
    Agente personalizado que orquesta el pipeline ETL completo para UN
    archivo, sin depender de ningún framework de grafos. Ejecuta cada fase
    de forma secuencial sobre un estado compartido (diccionario) y usa las
    funciones de enrutamiento del Bloque 7 (ruta_post_*) para decidir si
    continúa al siguiente paso o corta la cadena hacia el manejo de errores.
    """

    def __init__(self):
        # (nombre, función_del_paso, función_de_ruta) en orden de ejecución.
        self.pasos = [
            ("lectura",        paso_lectura,        ruta_post_lectura),
            ("mapeo",          paso_mapeo,           ruta_post_mapeo),
            ("transformacion", paso_transformacion,  ruta_post_transformacion),
            ("carga",          paso_carga,           ruta_post_carga),
        ]

    def ejecutar(self, estado_inicial: dict) -> dict:
        """Corre el flujo completo: deteccion → lectura → mapeo → transformacion → carga → finalizar."""
        estado = dict(estado_inicial)

        # Paso fijo de entrada (siempre ocurre, no tiene ruta condicional)
        estado.update(paso_deteccion(estado))

        for nombre, fn_paso, fn_ruta in self.pasos:
            estado.update(fn_paso(estado))
            if fn_ruta(estado) == "error":
                estado.update(paso_error(estado))
                return estado

        estado.update(paso_finalizar(estado))
        return estado


agente_etl = AgenteETL()

print("✅ Agente ETL personalizado ensamblado (clase AgenteETL, sin frameworks de grafos).")
print()
print("Flujo del agente:")
print("  deteccion → lectura →? mapeo →? transformacion →? carga →? finalizar")
print("                       ↘          ↘                 ↘         ↘")
print("                      error      error             error     error")


## Orquestador: ejecuta el agente ETL para todos los archivos pendientes

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ORQUESTADOR DEL AGENTE ETL
# Recorre los 4 volúmenes, recolecta los archivos pendientes y ejecuta el
# agente ETL (lectura → mapeo → transformación → carga a Delta) para cada uno.
# ══════════════════════════════════════════════════════════════════════════

print("="*70)
print("🔧 AGENTE ETL — TRANSFORMACIÓN Y CARGA A DELTA — INICIANDO")
print("="*70)

VOLUMENES = [
    (RAW_PATH_ESPAC,   "ESPAC"),
    (RAW_PATH_SIPA,    "SIPA"),
    (RAW_PATH_FAOSTAT, "FAOSTAT"),
    (RAW_PATH_AEBE,    "AEBE"),
]

pendientes = []
for vol_path, vol_label in VOLUMENES:
    try:
        for f in dbutils.fs.ls(vol_path):
            n = f.name
            if not n.lower().endswith((".xlsx", ".xls", ".csv", ".json", ".pdf")):
                continue
            rp = f.path
            cp = (rp.replace("dbfs:", "") if rp.startswith("dbfs:/Volumes")
                  else rp.replace("dbfs:", "/dbfs") if rp.startswith("dbfs:/") else rp)
            pendientes.append({"name": n, "path": cp, "vol": vol_label})
    except Exception as e:
        print(f"  ⚠️ Listado {vol_label}: {e}")

print(f"\n📂 Archivos pendientes encontrados: {len(pendientes)}")

resultados_etl = []

for archivo in pendientes:
    print(f"\n{'#'*70}")
    print(f"▶ PROCESANDO: {archivo['name']}")
    print(f"{'#'*70}")

    estado_inicial: ETLState = {
        "file_name": archivo["name"], "local_path": archivo["path"],
        "fuente": "", "inicio_ts": datetime.now().isoformat(),
        "df_columnas": [], "total_filas": 0, "ts_fin_lectura": "", "error_lectura": None,
        "tabla_destino": "", "cols_double": [], "cols_integer": [],
        "llm_response": "{}", "llamadas_api": 0, "error_mapeo": None,
        "registros_validos": 0, "registros_duplicados": 0,
        "ts_fin_transformacion": "", "error_transform": None, "temp_view_name": "",
        "tabla_completa": "", "ts_fin_carga": "", "error_carga": None,
        "tiempo_segundos": 0.0, "status": "PENDIENTE", "error_final": None,
    }

    try:
        estado_final = ejecutar_agente_etl(estado_inicial)
        resultados_etl.append(estado_final)

        icono = "✅" if estado_final["status"] == "PROCESADO" else "❌"
        print(f"\n{icono} {archivo['name']} → {estado_final['status']}")
        print(f"   Tabla destino: {estado_final.get('tabla_completa', 'N/A')} | "
              f"Registros válidos: {estado_final.get('registros_validos', 0)} | "
              f"Tiempo: {estado_final.get('tiempo_segundos', 0):.1f}s")

    except Exception as e:
        print(f"\n❌ Error fatal procesando {archivo['name']}: {e}")
        resultados_etl.append({"file_name": archivo["name"], "status": "ERROR",
                                "registros_validos": 0, "error_final": str(e)})

# ── RESUMEN FINAL ───────────────────────────────────────────────────────────
exitosos = sum(1 for r in resultados_etl if r.get("status") == "PROCESADO")
errores = sum(1 for r in resultados_etl if r.get("status") == "ERROR")
registros_procesados = sum(r.get("registros_validos", 0) for r in resultados_etl)
registros_error = errores

print(f"\n{'='*70}")
print("🎉 AGENTE ETL FINALIZADO")
print(f"{'='*70}")
print(f"  Archivos procesados: {len(resultados_etl)}")
print(f"  Exitosos: {exitosos} | Con error: {errores}")
print(f"  Total registros válidos cargados: {registros_procesados}")
print(f"{'='*70}")

# ========== EXPORTAR TASK VALUES (OBLIGATORIO PARA EL JOB) ==========
status_final_job = "success" if (len(resultados_etl) > 0 and errores == 0) else "error"
dbutils.jobs.taskValues.set(key="status", value=status_final_job)
dbutils.jobs.taskValues.set(key="registros_procesados", value=registros_procesados)
print(f"✅ Task Values exportados: status={status_final_job}, registros_procesados={registros_procesados}")
